# M0: AV 3.0 Pipeline Overview

**Pipeline Position:** Overview & Reference Module  
**Input S3 Path:** N/A (reference only)  
**Output S3 Path:** N/A (reference only)  
**Source Repo:** [NVIDIA Cosmos Cookbook](https://github.com/NVIDIA/Cosmos/tree/main/cosmos-cookbook)  
**Instance:** ml.t3.medium (CPU only)

## Pipeline Architecture

This lab implements the **8-stage** end-to-end Physical AI data pipeline from the
[AWS + NVIDIA AV 3.0 blog](https://aws.amazon.com/blogs/industries/building-an-end-to-end-physical-ai-data-pipeline-for-autonomous-vehicle-3-0-on-aws-with-nvidia/),
plus a few supplementary modules (M5, M9, M11) that extend it. Everything derives
from two sources in the shared S3 bucket: the **nuScenes-mini dataset** and the
**pre-cached model weights** (`model-cache/`). Arrows show data flow.

```
 SHARED S3  (av30lab-shared-data-<account>)
 ┌─────────────────────────────────────┐   ┌──────────────────────────────────────┐
 │ datasets/nuscenes-mini/             │   │ model-cache/  (pre-cached GPU models) │
 │   metadata tables + CAM_FRONT images│   │   cosmos-reason1/    → M2              │
 └───────┬───────────────────┬─────────┘   │   cosmos-transfer2.5/→ M4              │
   M1     │                   │  M10        │   cosmos-predict2.5/ → M5              │
   reads  ▼                   ▼  reads CAM   │   alpamayo-1.5/      → M6              │
 ┌────────────┐         ┌──────────────────┐└──────────────────────────────────────┘
 │ M1         │Stage 1-2│ M10  Stage 6     │        (each GPU module s3-syncs its
 │ Data       │         │ Nerfstudio 3D    │         model folder at run time)
 │ Explore    │         │ Reconstruction   │
 │ CPU        │         │ g5.xlarge        │
 └─────┬──────┘         └──────────────────┘
       │ m1/
       ▼
 ┌────────────┐  m2/    ┌────────────┐  m3/     ┌───────────────┐
 │ M2 Stage 3 │ caption │ M3 Stage 3 │ curated  │ (M3 fans out  │
 │ Cosmos     │────────▶│ Cosmos     │─────────▶│  to 4 modules │
 │ Reason     │         │ Curator    │          │  below)       │
 │ Captioning │         │ (filter)   │          └──────┬────────┘
 │ g5.12xlarge│         │ g5.12xlarge│                 │
 └─────┬──────┘         └────────────┘                 │
       │ m2/captions.json                              │
       ▼                                               │
 ┌────────────┐ Stage 4                                │
 │ M8         │ Search & Indexing                      │
 │ OpenSearch │ (AWS-native Cosmos                     │
 │ Semantic   │  Dataset Search)                       │
 │ Search CPU │                                        │
 └────────────┘                                        │
        ┌──────────────┬──────────────┬────────────────┘
        ▼              ▼              ▼              ▼
  ┌──────────┐  ┌──────────┐  ┌────────────┐  ┌──────────────┐
  │ M4       │  │ M5       │  │ M6         │  │ M9           │
  │ Stage 5  │  │ Stage 5  │  │ Stage 7    │  │ Stages 3/5/7 │
  │ Cosmos   │  │ (ext)    │  │ Alpamayo   │  │ (ext)        │
  │ Transfer │  │ Cosmos   │  │ VLA (train │  │ HyperPod     │
  │ (Weather │  │ Predict  │  │ /inference)│  │ Distributed  │
  │  Aug)    │  │ (Scenario│  │            │  │ Training     │
  │ p4d.24xl │  │  Gen)    │  │ p4d.24xl   │  │ p4d (concept)│
  └──────────┘  │ p4d.24xl │  └─────┬──────┘  └──────────────┘
                └──────────┘        │ m6/ policy
                                    ▼
                              ┌────────────┐ Stage 8
                              │ M7         │ Software-in-the-Loop
                              │ AlpaSim    │ Testing (closed-loop)
                              │ Eval       │
                              │ g5.12xlarge│
                              └────────────┘

 Orchestration:  ┌──────────────────┐
                 │ M11 (ext)        │ reads m1/ , runs M1→M2→M3→M4
                 │ Pipeline Autom.  │ as one SageMaker Pipeline
                 │ CPU              │
                 └──────────────────┘
```

**Blog stage → module mapping** (blog calls these 8 items *stages*, grouped into 4
*phases*: Ingest, Data Processing, Train, Validate):

| Blog Stage | Module(s) |
|-----------|-----------|
| 1. Ingest to cloud | M1 |
| 2. Data quality & sensor extraction | M1 |
| 3. Data curation (Cosmos Reason + Curator) | M2, M3 |
| 4. Search and indexing (OpenSearch) | M8 |
| 5. Data augmentation (Cosmos Transfer) | M4 |
| 6. Neural reconstruction (NuRec → Nerfstudio here) | M10 |
| 7. Model training (Alpamayo) | M6 |
| 8. Software-in-the-loop testing (AlpaSim) | M7 |
| *extension* — synthetic scenario gen (Cosmos Predict) | M5 |
| *extension* — distributed training scale-up (HyperPod) | M9 |
| *extension* — pipeline orchestration | M11 |

**Reading the diagram**
- **Two sources** in the shared bucket: `datasets/nuscenes-mini/` and `model-cache/`.
- **Core path:** M1 → M2 → M3 (ingest → caption → curate).
- **M3 fans out** to M4, M5, M6, M9 — each consumes `m3/`.
- **Model cache:** M2/M4/M5/M6 each `aws s3 sync` their model folder from
  `model-cache/` at run time (Cosmos Reason 1 / Transfer 2.5 / Predict 2.5 / Alpamayo 1.5).
- **M8** branches off **M2** (`m2/captions.json`); **M7** consumes **M6** (`m6/`).
- **M10** reads the **nuScenes CAM_FRONT images directly** (independent branch).
- **M11** orchestrates the M1→M4 sub-pipeline as one SageMaker Pipeline.


In [ ]:
"""NVIDIA AV Tool Availability for AV 3.0 Blueprint Lab"""
import pandas as pd

tools_data = [
    {
        "Tool": "Cosmos Predict 2.5",
        "Version": "2.5",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos",
        "How to Access": "HuggingFace: nvidia/Cosmos-Predict2-2B / 14B"
    },
    {
        "Tool": "Cosmos Transfer 2.5",
        "Version": "2.5",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos",
        "How to Access": "HuggingFace: nvidia/Cosmos-Transfer2-2B / 14B"
    },
    {
        "Tool": "Cosmos Reason 1",
        "Version": "1.0",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos/tree/main/cosmos-cookbook/post_training/reason1",
        "How to Access": "HuggingFace: nvidia/Cosmos-Reason1-7B"
    },
    {
        "Tool": "Cosmos Reason 2",
        "Version": "2.0",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos/tree/main/cosmos-cookbook/post_training/reason2",
        "How to Access": "HuggingFace: nvidia/Cosmos-Reason2-2B / 7B"
    },
    {
        "Tool": "Cosmos Curator (NeMo Curator)",
        "Version": "0.8+",
        "Status": "Available",
        "License": "Apache 2.0",
        "Source URL": "https://github.com/NVIDIA/NeMo-Curator",
        "How to Access": "pip install nemo-curator"
    },
    {
        "Tool": "Alpamayo 1.5 (VLA)",
        "Version": "1.5",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVlabs/alpamayo1.5",
        "How to Access": "HuggingFace: nvidia/Alpamayo-1.5-10B (used in M6)"
    },
    {
        "Tool": "AlpaSim (closed-loop sim)",
        "Version": "1.0",
        "Status": "Preview",
        "License": "NVIDIA Proprietary",
        "Source URL": "https://github.com/NVlabs/alpasim",
        "How to Access": "NVIDIA Developer Program (used in M7)"
    },
    {
        "Tool": "Physical AI Datasets (nuScenes)",
        "Version": "1.0",
        "Status": "Available",
        "License": "CC BY-NC-SA 4.0",
        "Source URL": "https://www.nuscenes.org/",
        "How to Access": "Pre-loaded in shared S3 bucket (mini split)"
    },
    {
        "Tool": "Cosmos Tokenizer",
        "Version": "1.0",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos-Tokenizer",
        "How to Access": "HuggingFace: nvidia/Cosmos-Tokenizer-*"
    },
    {
        "Tool": "DRIVE Sim (Omniverse)",
        "Version": "2024.2",
        "Status": "Enterprise Only",
        "License": "NVIDIA Enterprise",
        "Source URL": "https://developer.nvidia.com/drive/simulation",
        "How to Access": "NVIDIA Enterprise subscription required"
    },
    {
        "Tool": "Cosmos World Foundation Model (WFM)",
        "Version": "1.0",
        "Status": "Available",
        "License": "NVIDIA Open Model License",
        "Source URL": "https://github.com/NVIDIA/Cosmos",
        "How to Access": "HuggingFace: nvidia/Cosmos-Predict1-*"
    }
]

df_tools = pd.DataFrame(tools_data)
print("=" * 80)
print("NVIDIA AV 3.0 Tool Availability Matrix")
print("=" * 80)
df_tools.style.set_properties(**{'text-align': 'left'})
df_tools

## AWS Service Mapping per Module (blog 8-stage)

| Module | Pipeline Stage | AWS Service | Instance Type | Purpose |
|--------|---------------|-------------|---------------|---------|
| M1 | Data Collection (Stage 1-2) | S3 + SageMaker Studio | ml.t3.medium | Browse & explore nuScenes-mini |
| M2 | AV Captioning (Stage 3) | SageMaker JupyterLab | ml.g5.12xlarge | Cosmos Reason 1 inference |
| M3 | Data Curation (Stage 3) | SageMaker JupyterLab | ml.g5.12xlarge | Quality filtering & dedup of M2 captions |
| M4 | Weather Augmentation (Stage 5) | SageMaker JupyterLab | ml.p4d.24xlarge | Cosmos Transfer 2.5 |
| M5 | Scenario Generation (Stage 5, ext) | SageMaker JupyterLab | ml.p4d.24xlarge | Cosmos Predict 2.5 |
| M6 | VLA Inference (Stage 7) | SageMaker JupyterLab | ml.p4d.24xlarge | Alpamayo 1.5 inference |
| M7 | Closed-Loop Eval (Stage 8) | SageMaker JupyterLab | **ml.t3.medium (CPU)** | Visualizes an AlpaSim eval; real sim runs on a GPU EC2 host |
| M8 | Semantic Search (Stage 4) | SageMaker + OpenSearch Serverless | ml.t3.medium | Video clip retrieval (Cosmos Dataset Search replacement) |
| M9 | Distributed Training (Stages 3/5/7, ext) | SageMaker Training | **ml.t3.medium notebook → ml.m5.xlarge×2 job** | Submits a real 2-node DDP job; HyperPod is conceptual |
| M10 | 3D Reconstruction (Stage 6) | SageMaker JupyterLab | ml.g5.xlarge | Nerfstudio neural radiance fields *(training cell known-limited)* |
| M11 | Orchestration | SageMaker Pipelines | **ml.t3.medium notebook → ml.m5.xlarge steps** | Automate M1→M4 as one SageMaker Pipeline |

\* **M7 and M9 notebooks are CPU** even though they're about GPU-scale ideas: M7
visualizes a closed-loop simulation the admin ran on a GPU EC2 host; M9 *submits* a
real 2-node `torch.distributed` job that runs on separate managed `ml.m5.xlarge`
instances. M9's HyperPod *cluster* is separate infrastructure — the notebook
demonstrates the distributed-training pattern, not a live HyperPod cluster.
See docs/HYPERPOD_M9.md and docs/ALPASIM_M7.md.

**Shared Infrastructure:**
- Shared data bucket: `s3://av30lab-shared-data-{account_id}/` (datasets + model cache)
- User workspace bucket: `s3://av30lab-user-workspace-{account_id}/users/{profile}/`
- IAM: SageMaker execution role with scoped S3, ECR, and CloudWatch access
- Network: SageMaker Studio domain in PublicInternetOnly mode
- GPU modules require the **SageMaker Distribution GPU** image (selected automatically
  when a GPU instance is chosen)


## Module Overview

| Module | Name | Instance | Blog Stage | Reads | Description |
|--------|------|----------|-------|-------|-------------|
| M0 | Pipeline Overview | ml.t3.medium | Overview | — | Architecture & tool reference |
| M1 | Data Exploration | ml.t3.medium | 1-2 | nuScenes-mini | Explore the nuScenes-mini dataset → `m1/` |
| M2 | Cosmos Reason Captioning | ml.g5.12xlarge | 3 | `m1/` | AV video captioning (Cosmos Reason 1) → `m2/` |
| M3 | Cosmos Curator | ml.g5.12xlarge | 3 | `m2/` | Data quality filtering & curation → `m3/` |
| M4 | Cosmos Transfer — Weather Aug | ml.p4d.24xlarge | 5 | `m3/` | Weather-augmented clips (Cosmos Transfer 2.5) → `m4/` |
| M5 | Cosmos Predict — Scenario Gen | ml.p4d.24xlarge | 5 (ext) | `m3/` | Synthetic traffic scenarios (Cosmos Predict 2.5) → `m5/` |
| M6 | Alpamayo VLA Inference | ml.p4d.24xlarge | 7 | `m3/` | Vision-Language-Action policy inference → `m6/` |
| M7 | AlpaSim Closed-Loop Eval | ml.t3.medium (CPU) | 8 | `m6/` | Visualizes a real AlpaSim eval (sim runs on a GPU EC2 host) → `m7/` |
| M8 | OpenSearch Semantic Search | ml.t3.medium | 4 | `m2/` | Video clip retrieval (embeddings + k-NN) → `m8/` |
| M9 | HyperPod Distributed Training | ml.t3.medium (CPU) | 3/5/7 (ext) | `m3/` | Submits a real 2-node DDP job on `ml.m5.xlarge`×2 → `m9/` |
| M10 | Nerfstudio 3D Reconstruction | ml.g5.xlarge | 6 | nuScenes CAM_FRONT | Neural radiance field reconstruction → `m10/` *(training cell known-limited)* |
| M11 | Pipeline Automation | ml.t3.medium | — (ext) | `m1/` | Orchestrate M1→M4 as one SageMaker Pipeline → `m11/` |

\* M7/M9 notebooks are **CPU**: M7 visualizes an admin-run AlpaSim eval; M9 submits a
real 2-node `torch.distributed` job to separate `ml.m5.xlarge` instances. M9's
HyperPod cluster is separate infrastructure (demonstrated conceptually).

**GPU notebook modules (M2–M6, M10)** launch on GPU instances; the workshop selects
the matching SageMaker Distribution **GPU image** automatically. CPU modules (M0, M1,
**M7**, M8, **M9**, M11) run on `ml.t3.medium`. (M7's real simulation and M9's real
training job run on GPU/managed compute *outside* the notebook.)

**Cost note:** the biggest drivers are the `ml.p4d.24xlarge` modules (M4/M5/M6,
~$37/hr). M9's managed training job (`ml.m5.xlarge`×2) and M11's pipeline steps are
CPU and cost cents per run. Idle apps auto-shut down after 3 hours; stop your space
when done to avoid charges.

---

*Proceed to **M1_Data_Exploration.ipynb** to begin the hands-on pipeline.*
